In [ ]:
# Configure every external directory here before executing the notebook.
from pathlib import Path
import os
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent
RAW_DIR = Path(os.environ.get('RIDESHARE_RAW_DIR', str(PROJECT_DIR / 'data/raw/fhvhv')))
PROCESSED_DIR = Path(os.environ.get('RIDESHARE_PROCESSED_DIR', str(PROJECT_DIR / 'data/processed')))
ZONE_DIR = Path(os.environ.get('RIDESHARE_ZONE_DIR', str(PROJECT_DIR / 'data/reference/taxi_zones')))
WEATHER_DIR = Path(os.environ.get('RIDESHARE_WEATHER_DIR', str(PROJECT_DIR / 'data/reference/weather_hourly')))
ANALYSIS_DIR = PROJECT_DIR / 'outputs/demand_analysis'
# Spark users: configure JAVA_HOME and, on Windows when required, HADOOP_HOME externally.

import os
import glob
import json
import hashlib
import calendar
import gc
from pathlib import Path
from datetime import datetime, timezone

# Use the Java/Hadoop installation configured in your environment.
# Use the Java/Hadoop installation configured in your environment.

from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel

spark = (SparkSession.builder.appName('TLC_FHVHV_Selected_Repairs')
         .master('local[4]')
         .config('spark.driver.memory', '6g')
         .config('spark.sql.shuffle.partitions', '64')
         .config('spark.sql.parquet.enableVectorizedReader', 'false')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)
print('Spark session timezone:', spark.conf.get('spark.sql.session.timeZone'))

# 02c. Validate selected staged repairs

Rebuild and validate four selected monthly outputs: OD March/April 2024 and dropoff August/October 2025. Run one month at a time after other Spark preparation sessions have stopped. Cleaning, metric names, filenames and recorded platform scope are preserved.

Outputs are staged under the configured processed directory's `repair_2024_2025` subfolder. Existing analytical files are not overwritten. A successful reconciliation writes a versioned validation record. Stop on the first failed assertion; no promotion is performed.


In [ ]:
RAW_DATA_PATH = RAW_DIR
OUTPUT_PATH = PROCESSED_DIR
REPAIR_PATH = OUTPUT_PATH / 'repair_2024_2025'
REPAIR_PATH.mkdir(parents=True, exist_ok=True)
POLICY_DATE = '2025-01-05'
REPAIR_VERSION = 'selected-repairs-v1-legacy-dropoff-date'
TASKS = [('od', 2024, 3), ('od', 2024, 4),
         ('dropoff', 2025, 8), ('dropoff', 2025, 10)]
STEMS = {'od': 'fhvhv_by_OD_hour', 'dropoff': 'fhvhv_by_DOzone_hour'}

for family, year, month in TASKS:
    source = RAW_DATA_PATH / f'fhvhv_tripdata_{year}-{month:02d}.parquet'
    if not source.is_file() or source.stat().st_size == 0:
        raise FileNotFoundError(f'Raw input missing or empty: {source}')
    print(family, f'{year}-{month:02d}', '->', REPAIR_PATH / f'{STEMS[family]}_{year}_{month:02d}.parquet')

## Cleaning and features

Filters are copied from the existing 2024/2025 processing notebook, including
speed based on reported trip_time and duration based on timestamp differences.
No new platform filter or airport-fee imputation is introduced. Null CBD fees
are set to zero as in the original workflow. Monthly file contents must belong
to the designated pickup month; unexpected rows cause a validation failure.
Disk persistence avoids repeatedly recomputing cleaning within a repair month.

In [ ]:
def prepare_month(source):
    df = spark.read.parquet(str(source))
    if 'cbd_congestion_fee' not in df.columns:
        # Earlier schemas lack this field; add null before the original cleaning.
        df = df.withColumn('cbd_congestion_fee', F.lit(None).cast('double'))
    df = (df
        .withColumn('trip_time_cal', F.unix_timestamp('dropoff_datetime') - F.unix_timestamp('pickup_datetime'))
        .withColumn('trip_speed', (F.col('trip_miles') / F.col('trip_time')) * 3600))
    df_clean = (df.dropDuplicates()
        .filter(F.col('trip_miles') > 0)
        .filter(F.col('trip_time_cal') > 0)
        .filter(F.col('base_passenger_fare') > 0)
        .filter(F.col('trip_speed') > 1)
        .filter(F.col('driver_pay') > 0)
        .filter(F.col('pickup_datetime') >= F.col('request_datetime'))
        .filter(F.col('trip_miles') <= 200)
        .filter(F.col('trip_time_cal') <= 10800)
        .filter(F.col('trip_speed') <= 120)
        .withColumn('cbd_congestion_fee', F.coalesce(F.col('cbd_congestion_fee'), F.lit(0.0))))
    return (df_clean
        .withColumn('year', F.year('pickup_datetime'))
        .withColumn('month', F.month('pickup_datetime'))
        .withColumn('week', F.weekofyear('pickup_datetime'))
        .withColumn('date', F.to_date('pickup_datetime'))
        .withColumn('hour', F.hour('pickup_datetime'))
        .withColumn('do_hour', F.hour('dropoff_datetime'))
        .withColumn('day_of_week', F.dayofweek('pickup_datetime'))
        .withColumn('is_weekend', F.when(F.col('day_of_week').isin(1, 7), 1).otherwise(0))
        .withColumn('policy_period', F.when(F.col('pickup_datetime') < F.lit(POLICY_DATE), 'pre_policy').otherwise('post_policy'))
        .withColumn('wait_time_sec', F.unix_timestamp('pickup_datetime') - F.unix_timestamp('request_datetime'))
        .withColumn('trip_duration_min', F.col('trip_time_cal') / 60)
        .withColumn('total_fare', F.col('base_passenger_fare') + F.col('tolls') + F.col('tips') +
                    F.col('congestion_surcharge') + F.col('sales_tax') + F.col('bcf') +
                    F.col('airport_fee') + F.col('cbd_congestion_fee'))
        .withColumn('tnc_comp', F.col('base_passenger_fare') + F.col('tips') - F.col('driver_pay')))

In [ ]:
def compute_metrics(df, group_cols):
    """
    Aggregate all metrics using aggregate-first approach.
    Group by group_cols and compute all metrics.
    """
    return df.groupBy(group_cols).agg(
        # Demand
        F.count('*').alias('trip_count'),

        # Intermediate sums for rate calculations
        F.sum('wait_time_sec').alias('_sum_wait_time'),
        F.sum('base_passenger_fare').alias('_sum_base_fare'),
        F.sum('total_fare').alias('_sum_total_fare'),
        F.sum('driver_pay').alias('_sum_driver_pay'),
        F.sum('tnc_comp').alias('_sum_tnc_comp'),
        F.sum('trip_miles').alias('_sum_trip_miles'),
        F.sum('trip_duration_min').alias('_sum_trip_duration_min')
    ).withColumn(
        'avg_wait_time_sec',     F.round(F.col('_sum_wait_time') / F.col('trip_count'), 2)
    ).withColumn(
        'base_fare_per_mile',    F.round(F.col('_sum_base_fare') / F.col('_sum_trip_miles'), 4)
    ).withColumn(
        'base_fare_per_min',     F.round(F.col('_sum_base_fare') / F.col('_sum_trip_duration_min'), 4)
    ).withColumn(
        'base_fare_per_trip',    F.round(F.col('_sum_base_fare') / F.col('trip_count'), 4)
    ).withColumn(
        'total_fare_per_trip',   F.round(F.col('_sum_total_fare') / F.col('trip_count'), 4)
    ).withColumn(
        'driver_pay_per_mile',   F.round(F.col('_sum_driver_pay') / F.col('_sum_trip_miles'), 4)
    ).withColumn(
        'driver_pay_per_min',    F.round(F.col('_sum_driver_pay') / F.col('_sum_trip_duration_min'), 4)
    ).withColumn(
        'driver_pay_per_trip',   F.round(F.col('_sum_driver_pay') / F.col('trip_count'), 4)
    ).withColumn(
        'tnc_commission_rate',   F.round(F.col('_sum_tnc_comp') / F.col('_sum_base_fare'), 4)
    ).drop(
        '_sum_wait_time', '_sum_base_fare', '_sum_total_fare',
        '_sum_driver_pay', '_sum_tnc_comp', '_sum_trip_miles', '_sum_trip_duration_min'
    )

In [ ]:
def aggregate_od(df):
    keys = ['PULocationID', 'DOLocationID', 'date', 'year', 'month', 'week',
            'day_of_week', 'is_weekend', 'hour', 'policy_period']
    return (df.groupBy(keys).agg(
        F.count('*').alias('trip_count'),
        F.sum('base_passenger_fare').alias('_sum_base_fare'),
        F.sum('total_fare').alias('_sum_total_fare'),
        F.sum('trip_miles').alias('_sum_trip_miles'),
        F.sum('trip_duration_min').alias('_sum_trip_duration_min'),
        F.sum('cbd_congestion_fee').alias('total_cbd_fee'))
        .withColumn('avg_base_fare', F.round(F.col('_sum_base_fare') / F.col('trip_count'), 4))
        .withColumn('avg_total_fare', F.round(F.col('_sum_total_fare') / F.col('trip_count'), 4))
        .withColumn('avg_trip_miles', F.round(F.col('_sum_trip_miles') / F.col('trip_count'), 4))
        .withColumn('avg_duration_min', F.round(F.col('_sum_trip_duration_min') / F.col('trip_count'), 4))
        .drop('_sum_base_fare', '_sum_total_fare', '_sum_trip_miles', '_sum_trip_duration_min'))

def total_trips(df):
    return int(df.agg(F.sum('trip_count').alias('n')).first()['n'] or 0)

def compare_counts(left, right, keys):
    a = left.groupBy(keys).agg(F.sum('trip_count').alias('left_count'))
    b = right.groupBy(keys).agg(F.sum('trip_count').alias('right_count'))
    diff = a.join(b, keys, 'full').fillna(0, subset=['left_count', 'right_count'])
    return diff.filter(F.col('left_count') != F.col('right_count')).count()

def contract_for(source, family, year, month):
    stat = source.stat()
    return {'version': REPAIR_VERSION, 'family': family, 'year': year, 'month': month,
            'raw_file': source.name, 'raw_bytes': stat.st_size, 'raw_mtime_ns': stat.st_mtime_ns,
            'time_basis': 'pickup_date_dropoff_hour' if family == 'dropoff' else 'pickup_date_pickup_hour'}

def ready(output, contract):
    marker = output / '_repair_validation.json'
    if not (marker.is_file() and (output / '_SUCCESS').is_file() and list(output.glob('*.parquet'))):
        return False
    try:
        data = json.loads(marker.read_text(encoding='utf-8'))
        return data['passed'] and data['contract'] == contract
    except (ValueError, KeyError):
        return False

## Execute repairs and validate

For every month, validate against the cleaned raw-month count and the existing
pickup table. OD is also reconciled by pickup zone/date/hour; drop-off is
reconciled with OD by destination/pickup date, across hours. The staged output
must have complete calendar-date coverage, positive counts and unique keys.
Any failed assertion stops processing and prevents a validated checkpoint.
Neither successful writes nor `_SUCCESS` alone establish validation.

In [ ]:
summary = []
for family, year, month in TASKS:
    source = RAW_DATA_PATH / f'fhvhv_tripdata_{year}-{month:02d}.parquet'
    output = REPAIR_PATH / f'{STEMS[family]}_{year}_{month:02d}.parquet'
    contract = contract_for(source, family, year, month)
    if ready(output, contract):
        print('Already validated, skipping:', output)
        summary.append(json.loads((output / '_repair_validation.json').read_text(encoding='utf-8')))
        continue
    print('Preparing:', source)
    df = prepare_month(source).persist(StorageLevel.DISK_ONLY)
    try:
        clean_count = df.count()
        assert clean_count > 0, 'Cleaned month is empty'
        assert df.filter((F.col('year') != year) | (F.col('month') != month)).count() == 0, 'Unexpected pickup dates in raw month'
        pickup = spark.read.parquet(str(OUTPUT_PATH / f'fhvhv_by_zone_hour_{year}_{month:02d}.parquet'))
        pickup_total = total_trips(pickup)
        assert clean_count == pickup_total, f'Cleaning differs from pickup reference: {clean_count:,} vs {pickup_total:,}'
        if family == 'od':
            agg = aggregate_od(df)
            keys = ['PULocationID', 'DOLocationID', 'date', 'hour']
            template_path = OUTPUT_PATH / 'fhvhv_by_OD_hour_2024_01.parquet'
        else:
            groups = ['DOLocationID', 'date', 'year', 'month', 'week', 'day_of_week', 'is_weekend', 'do_hour', 'policy_period']
            agg = compute_metrics(df, groups)
            keys = ['DOLocationID', 'date', 'do_hour']
            template_path = OUTPUT_PATH / 'fhvhv_by_DOzone_hour_2024_01.parquet'
        # Match the established output column order and Spark data types.
        schema = spark.read.parquet(str(template_path)).schema
        agg = agg.select(*[F.col(field.name).cast(field.dataType).alias(field.name) for field in schema])
        agg.write.mode('overwrite').parquet(str(output))
        candidate = spark.read.parquet(str(output))
        candidate_total = total_trips(candidate)
        assert candidate_total == clean_count, 'Written aggregate count does not reconcile with cleaned raw month'
        null_keys = None
        for name in keys:
            term = F.col(name).isNull()
            null_keys = term if null_keys is None else null_keys | term
        assert candidate.filter(null_keys | F.col('trip_count').isNull() | (F.col('trip_count') <= 0)).count() == 0, 'Invalid keys/counts'
        assert candidate.groupBy(keys).count().filter(F.col('count') > 1).count() == 0, 'Duplicate keys'
        assert candidate.select('date').distinct().count() == calendar.monthrange(year, month)[1], 'Missing calendar dates'
        if family == 'od':
            mismatches = compare_counts(candidate, pickup, ['PULocationID', 'date', 'hour'])
        else:
            od = spark.read.parquet(str(OUTPUT_PATH / f'fhvhv_by_OD_hour_{year}_{month:02d}.parquet'))
            assert total_trips(od) == clean_count, 'OD reference does not reconcile with cleaned month'
            mismatches = compare_counts(candidate, od, ['DOLocationID', 'date'])
        assert mismatches == 0, f'Reference mismatch: {mismatches} keys'
        result = {'contract': contract, 'passed': True, 'cleaned_trips': clean_count,
                  'pickup_reference_trips': pickup_total, 'candidate_trips': candidate_total,
                  'mismatched_keys': mismatches, 'validated_utc': datetime.now(timezone.utc).isoformat()}
        (output / '_repair_validation.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
        summary.append(result)
        (REPAIR_PATH / 'repair_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
        print('VALIDATED:', output, f'| trips: {candidate_total:,}')
    finally:
        df.unpersist(blocking=True)
        gc.collect()

## Review staged candidates

Inspect `repair_summary.json` and four successful `_repair_validation.json` records. Count agreement is necessary but does not validate every fare/fee value. Keep staged candidates separate from originals and never combine both copies in one monthly analysis. Promotion requires a separate reviewed decision; this notebook contains no replacement step.


In [ ]:
spark.stop()
print('Repair Spark session stopped. Review staged results before replacement.')

## Release the Spark session
Run after successful validation, before starting another preparation notebook.


In [ ]:
spark.stop()
